# IoT Telemetry Pipeline — End-to-End Showcase

This notebook walks through the complete medallion pipeline in a single narrative:

1. **Bronze → Silver → Gold Evolution** — how data transforms at each layer
2. **Anomaly Visualization** — where anomalies occur, their magnitude, and sensor breakdowns
3. **Device Health Over Time** — tracking fleet health trends and identifying at-risk devices

---

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pyspark.sql.functions import (
    col, count, avg, stddev, countDistinct, when, lit,
    sum as spark_sum, min as spark_min, max as spark_max,
    unix_timestamp, window, hour,
)
from delta.tables import DeltaTable

from pipeline.common.utils import load_config, get_spark_session

config = load_config()
spark = get_spark_session(config)

paths = config["paths"]
gold_base = paths["delta_gold"]

---
## Part 1: Bronze → Silver → Gold Evolution

The medallion architecture progressively refines raw telemetry into
analytics-ready datasets. Each layer adds governance, enrichment,
or aggregation — and nothing is silently lost.

In [ ]:
bronze_df = spark.read.format("delta").load(paths["delta_bronze"])
silver_df = spark.read.format("delta").load(paths["delta_silver"])

gold_dfs = {}
for t in ["device_summary", "anomaly_summary", "device_health", "ml_features"]:
    try:
        gold_dfs[t] = spark.read.format("delta").load(str(Path(gold_base) / t))
    except Exception:
        gold_dfs[t] = None

bronze_count = bronze_df.count()
silver_count = silver_df.count()
gold_counts = {k: v.count() if v else 0 for k, v in gold_dfs.items()}
gold_total = sum(gold_counts.values())

print(f"Bronze:  {bronze_count:,} rows")
print(f"Silver:  {silver_count:,} rows  ({bronze_count - silver_count:,} cleaned out)")
print(f"Gold:    {gold_total:,} rows  (across {len(gold_counts)} tables)")
for name, c in gold_counts.items():
    print(f"  └─ {name}: {c:,}")

### Record Funnel & Schema Growth

The funnel shows how many records survive each governance step, while
the column chart shows how the schema grows richer at each layer.

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Record Funnel", "Schema Width (columns)"],
    specs=[[{"type": "funnel"}, {"type": "bar"}]],
)

fig.add_trace(go.Funnel(
    y=["Bronze (Raw)", "Silver (Clean)", "Gold (Aggregated)"],
    x=[bronze_count, silver_count, gold_total],
    textinfo="value+percent initial",
    marker=dict(color=["#cd7f32", "#c0c0c0", "#ffd700"]),
), row=1, col=1)

layers = ["Bronze", "Silver", "Gold\nsummary", "Gold\nhealth", "Gold\nML"]
col_counts = [
    len(bronze_df.columns),
    len(silver_df.columns),
    len(gold_dfs["device_summary"].columns) if gold_dfs["device_summary"] else 0,
    len(gold_dfs["device_health"].columns) if gold_dfs["device_health"] else 0,
    len(gold_dfs["ml_features"].columns) if gold_dfs["ml_features"] else 0,
]
fig.add_trace(go.Bar(
    x=layers, y=col_counts,
    marker_color=["#cd7f32", "#c0c0c0", "#ffd700", "#daa520", "#b8860b"],
    text=col_counts, textposition="outside",
), row=1, col=2)

fig.update_layout(
    title="Medallion Architecture: Data Evolution",
    template="plotly_white", height=420, showlegend=False,
)
fig.show()

### What Each Layer Adds and Removes

Tracing exactly which columns appear and disappear as data flows through the pipeline.

In [ ]:
bronze_cols = set(bronze_df.columns)
silver_cols = set(silver_df.columns)

print("Bronze → Silver")
print(f"  Added:   {sorted(silver_cols - bronze_cols)}")
print(f"  Dropped: {sorted(bronze_cols - silver_cols)}")

if gold_dfs["ml_features"]:
    gold_ml_cols = set(gold_dfs["ml_features"].columns)
    print(f"\nSilver → Gold ML Features")
    print(f"  {len(gold_ml_cols)} columns — wide feature vector")
    print(f"  New columns: {sorted(gold_ml_cols - silver_cols)[:8]}...")

### Delta Lake Version History

Every write to a Delta table creates an auditable version entry.
This provides data lineage, rollback capability, and exactly-once guarantees.

In [ ]:
version_data = []
for name, path in [("Bronze", paths["delta_bronze"]), ("Silver", paths["delta_silver"])]:
    try:
        dt = DeltaTable.forPath(spark, path)
        hist = dt.history().select(
            "version", "timestamp", "operation",
            col("operationMetrics.numOutputRows").alias("rows_written"),
        ).toPandas()
        hist["layer"] = name
        version_data.append(hist)
    except Exception:
        pass

if version_data:
    all_hist = pd.concat(version_data, ignore_index=True)
    all_hist["rows_written"] = pd.to_numeric(all_hist["rows_written"], errors="coerce")
    all_hist = all_hist.sort_values("timestamp")

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=["Version History", "Rows Written per Commit"],
    )
    for layer in ["Bronze", "Silver"]:
        ldf = all_hist[all_hist["layer"] == layer]
        color = "#cd7f32" if layer == "Bronze" else "#808080"
        fig.add_trace(go.Scatter(
            x=ldf["timestamp"], y=ldf["version"],
            mode="lines+markers", name=layer, marker_color=color,
        ), row=1, col=1)
        fig.add_trace(go.Bar(
            x=ldf["timestamp"], y=ldf["rows_written"],
            name=f"{layer} rows", marker_color=color, showlegend=False,
        ), row=1, col=2)

    fig.update_layout(
        title="Delta Table Growth Over Time",
        template="plotly_white", height=380,
    )
    fig.show()

---
## Part 2: Anomaly Visualization

The Silver layer tags each record with per-sensor anomaly flags and z-scores.
This section visualizes the anomaly landscape across the entire fleet.

In [ ]:
silver_pdf = silver_df.toPandas()
anom_mask = silver_pdf["_is_anomaly"] == True

anom_count = anom_mask.sum()
clean_count = len(silver_pdf) - anom_count

type_counts = {
    "Temperature": (silver_pdf["_is_temp_anomaly"] == True).sum(),
    "Humidity":    (silver_pdf["_is_humidity_anomaly"] == True).sum(),
    "Pressure":    (silver_pdf["_is_pressure_anomaly"] == True).sum(),
}

print(f"Anomaly rate: {anom_count:,} / {len(silver_pdf):,} ({anom_count/len(silver_pdf)*100:.1f}%)")
for t, c in type_counts.items():
    print(f"  {t}: {c:,}")

### Anomaly Breakdown: Proportions and Per-Sensor Counts

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Clean vs Anomalous", "Anomalies by Sensor Type"],
    specs=[[{"type": "pie"}, {"type": "bar"}]],
)

fig.add_trace(go.Pie(
    labels=["Clean", "Anomalous"],
    values=[clean_count, anom_count],
    marker_colors=["#2ecc71", "#e74c3c"],
    hole=0.45, textinfo="label+percent",
), row=1, col=1)

fig.add_trace(go.Bar(
    x=list(type_counts.keys()),
    y=list(type_counts.values()),
    marker_color=["#e74c3c", "#3498db", "#9b59b6"],
    text=list(type_counts.values()), textposition="outside",
), row=1, col=2)

fig.update_layout(
    title="Anomaly Landscape",
    template="plotly_white", height=400, showlegend=False,
)
fig.show()

### Sensor Distributions with Anomaly Thresholds

Overlaying threshold boundaries on the raw sensor distributions reveals
how far out-of-range anomalous readings actually are.

In [ ]:
thresholds = {
    "temperature": {"min": -50, "max": 150, "flag": "_is_temp_anomaly"},
    "humidity":    {"min": 0,   "max": 100, "flag": "_is_humidity_anomaly"},
    "pressure":    {"min": 900, "max": 1100, "flag": "_is_pressure_anomaly"},
}

fig = make_subplots(rows=1, cols=3, subplot_titles=["Temperature", "Humidity", "Pressure"])

for i, (field, cfg) in enumerate(thresholds.items(), 1):
    clean_vals = silver_pdf[silver_pdf[cfg["flag"]] == False][field].dropna()
    anom_vals  = silver_pdf[silver_pdf[cfg["flag"]] == True][field].dropna()

    fig.add_trace(go.Histogram(
        x=clean_vals, name="Normal", marker_color="#2ecc71",
        opacity=0.7, showlegend=(i == 1),
    ), row=1, col=i)
    if len(anom_vals) > 0:
        fig.add_trace(go.Histogram(
            x=anom_vals, name="Anomalous", marker_color="#e74c3c",
            opacity=0.7, showlegend=(i == 1),
        ), row=1, col=i)

    fig.add_vline(x=cfg["min"], line_dash="dash", line_color="orange", row=1, col=i)
    fig.add_vline(x=cfg["max"], line_dash="dash", line_color="orange", row=1, col=i)

fig.update_layout(
    title="Sensor Distributions with Anomaly Thresholds (dashed = boundaries)",
    barmode="overlay", template="plotly_white", height=380,
)
fig.show()

### Z-Score Heatmap by Device

Z-scores reveal statistical outliers even when readings are within hard thresholds.
This heatmap shows the average |z-score| per device across all three sensors.

In [ ]:
zscore_cols = ["_temperature_zscore", "_humidity_zscore", "_pressure_zscore"]

device_zscores = silver_pdf.groupby("device_id")[zscore_cols].apply(
    lambda g: g.abs().mean()
).reset_index()
device_zscores = device_zscores.sort_values("_temperature_zscore", ascending=False).head(20)

z_matrix = device_zscores.set_index("device_id")[zscore_cols].values

fig = go.Figure(go.Heatmap(
    z=z_matrix,
    x=["Temperature", "Humidity", "Pressure"],
    y=device_zscores["device_id"].values,
    colorscale="YlOrRd",
    text=np.round(z_matrix, 2),
    texttemplate="%{text}",
    colorbar_title="Avg |z|",
))
fig.update_layout(
    title="Average |Z-Score| by Device (Top 20) — Higher = More Unusual",
    template="plotly_white", height=max(400, len(device_zscores) * 25 + 100),
)
fig.show()

### Anomaly Trends Over Time

Are anomalies evenly distributed or do they cluster? Temporal clustering
suggests environmental events rather than random sensor faults.

In [ ]:
silver_pdf["timestamp_dt"] = pd.to_datetime(silver_pdf["timestamp"], utc=True)
silver_pdf["minute"] = silver_pdf["timestamp_dt"].dt.floor("1min")

time_series = silver_pdf.groupby("minute").agg(
    total_events=("_is_anomaly", "count"),
    anomaly_count=("_is_anomaly", "sum"),
    temp_anom=("_is_temp_anomaly", "sum"),
    hum_anom=("_is_humidity_anomaly", "sum"),
    pres_anom=("_is_pressure_anomaly", "sum"),
).reset_index()
time_series["anomaly_rate"] = time_series["anomaly_count"] / time_series["total_events"]

fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
    subplot_titles=[
        "Event Throughput (events/min)",
        "Anomaly Rate Over Time",
        "Anomaly Breakdown by Type",
    ],
)

fig.add_trace(go.Scatter(
    x=time_series["minute"], y=time_series["total_events"],
    mode="lines", fill="tozeroy", line=dict(color="#3498db"),
    name="Events",
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=time_series["minute"], y=time_series["anomaly_rate"],
    mode="lines+markers", line=dict(color="#e74c3c"),
    marker=dict(size=4), name="Anomaly Rate",
), row=2, col=1)
fig.add_hline(y=0.1, line_dash="dot", line_color="gray",
              annotation_text="10% warning", row=2, col=1)

for col_name, label, color in [
    ("temp_anom", "Temperature", "#e74c3c"),
    ("hum_anom", "Humidity", "#3498db"),
    ("pres_anom", "Pressure", "#9b59b6"),
]:
    fig.add_trace(go.Bar(
        x=time_series["minute"], y=time_series[col_name],
        name=label, marker_color=color,
    ), row=3, col=1)

fig.update_layout(
    title="Temporal Anomaly Analysis",
    barmode="stack", template="plotly_white", height=700,
)
fig.show()

---
## Part 3: Device Health Over Time

The Gold layer computes a composite health score per device-window:

```
health = 0.4 × avg_quality + 0.3 × (1 − anomaly_rate) + 0.3 × (battery / 100)
```

Devices are classified into risk tiers: **healthy** (≥ 0.8), **warning** (≥ 0.5), **critical** (< 0.5).

In [ ]:
health_df = gold_dfs["device_health"]
if health_df is not None:
    health_pdf = health_df.toPandas()
    tier_colors = {"healthy": "#2ecc71", "warning": "#f39c12", "critical": "#e74c3c"}

    print(f"Total device-window entries: {len(health_pdf):,}")
    print(f"\nRisk Tier Distribution:")
    for tier, cnt in health_pdf["risk_tier"].value_counts().items():
        print(f"  {tier.capitalize()}: {cnt} ({cnt/len(health_pdf)*100:.1f}%)")
    print(f"\nHealth Score: mean={health_pdf['health_score'].mean():.3f}, "
          f"min={health_pdf['health_score'].min():.3f}, "
          f"max={health_pdf['health_score'].max():.3f}")
else:
    print("No device_health data available — run the Gold job first.")

### Fleet Health Distribution

In [ ]:
if health_df is not None:
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=["Health Score Distribution", "Risk Tier Breakdown"],
        specs=[[{"type": "histogram"}, {"type": "pie"}]],
    )

    for tier, color in tier_colors.items():
        subset = health_pdf[health_pdf["risk_tier"] == tier]
        if not subset.empty:
            fig.add_trace(go.Histogram(
                x=subset["health_score"], name=tier.capitalize(),
                marker_color=color, nbinsx=20,
            ), row=1, col=1)

    tier_counts = health_pdf["risk_tier"].value_counts()
    fig.add_trace(go.Pie(
        labels=[t.capitalize() for t in tier_counts.index],
        values=tier_counts.values,
        marker_colors=[tier_colors.get(t, "#888") for t in tier_counts.index],
        hole=0.4,
    ), row=1, col=2)

    fig.update_layout(
        title="Fleet Health Overview",
        template="plotly_white", height=400, barmode="stack",
    )
    fig.show()

### Device Health Rankings

Identifying the highest- and lowest-scoring devices reveals
which parts of the fleet need immediate attention.

In [ ]:
if health_df is not None:
    per_device = health_pdf.groupby("device_id").agg(
        avg_health=("health_score", "mean"),
        min_health=("health_score", "min"),
        avg_anomaly_rate=("anomaly_rate", "mean"),
        avg_battery=("avg_battery_level", "mean"),
        risk_tier=("risk_tier", lambda x: x.mode()[0]),
    ).reset_index().sort_values("avg_health")

    fig = go.Figure(go.Bar(
        y=per_device["device_id"],
        x=per_device["avg_health"],
        orientation="h",
        marker_color=per_device["risk_tier"].map(tier_colors),
        text=per_device["risk_tier"].str.capitalize(),
        textposition="inside",
    ))
    fig.add_vline(x=0.8, line_dash="dash", line_color="#2ecc71",
                  annotation_text="Healthy")
    fig.add_vline(x=0.5, line_dash="dash", line_color="#f39c12",
                  annotation_text="Warning")
    fig.update_layout(
        title="All Devices Ranked by Average Health Score",
        xaxis_title="Avg Health Score", xaxis_range=[0, 1.05],
        template="plotly_white",
        height=max(400, len(per_device) * 22 + 100),
    )
    fig.show()

### Health Score Decomposition — What Drives At-Risk Scores?

Breaking down the three health components (quality, anomaly-free rate, battery)
reveals whether devices are unhealthy because of data quality issues,
frequent anomalies, or low battery.

In [ ]:
if health_df is not None:
    at_risk = per_device.head(10).copy()

    at_risk["quality_contrib"]  = 0.4 * at_risk["avg_health"]  # approximate via score
    at_risk["anomaly_contrib"]  = 0.3 * (1 - at_risk["avg_anomaly_rate"])
    at_risk["battery_contrib"]  = 0.3 * (at_risk["avg_battery"] / 100)

    if "avg_quality_score" in health_pdf.columns:
        quality_by_device = health_pdf.groupby("device_id")["avg_quality_score"].mean()
        at_risk["quality_contrib"] = at_risk["device_id"].map(quality_by_device) * 0.4

    fig = go.Figure()
    for comp, label, color in [
        ("quality_contrib",  "Quality (40%)",      "#3498db"),
        ("anomaly_contrib",  "Anomaly-Free (30%)", "#e74c3c"),
        ("battery_contrib",  "Battery (30%)",      "#f39c12"),
    ]:
        fig.add_trace(go.Bar(
            y=at_risk["device_id"],
            x=at_risk[comp],
            name=label, orientation="h",
            marker_color=color,
        ))

    fig.update_layout(
        title="Health Score Decomposition — 10 Lowest-Scoring Devices",
        xaxis_title="Score Contribution",
        barmode="stack", template="plotly_white", height=420,
    )
    fig.show()

### Device Health Over Time (Windowed Trend)

Since Gold aggregates per time-window, we can see how each device's health
evolves over the streaming session.

In [ ]:
if health_df is not None and "window_start" in health_pdf.columns:
    health_pdf["window_start_dt"] = pd.to_datetime(health_pdf["window_start"], utc=True)

    # Pick the 5 most interesting devices (lowest average health)
    spotlight_devices = per_device.head(5)["device_id"].tolist()
    spotlight = health_pdf[health_pdf["device_id"].isin(spotlight_devices)]

    fig = go.Figure()
    for device_id in spotlight_devices:
        ddev = spotlight[spotlight["device_id"] == device_id].sort_values("window_start_dt")
        fig.add_trace(go.Scatter(
            x=ddev["window_start_dt"], y=ddev["health_score"],
            mode="lines+markers", name=device_id,
            marker=dict(size=6),
        ))

    fig.add_hline(y=0.8, line_dash="dash", line_color="#2ecc71",
                  annotation_text="Healthy threshold")
    fig.add_hline(y=0.5, line_dash="dash", line_color="#f39c12",
                  annotation_text="Warning threshold")

    fig.update_layout(
        title="Device Health Score Over Time (5 Most At-Risk Devices)",
        xaxis_title="Window Start", yaxis_title="Health Score",
        yaxis_range=[0, 1.05],
        template="plotly_white", height=450,
    )
    fig.show()
else:
    print("No windowed health data available for time-series visualization.")

### Anomaly Rate vs Battery Level — Early Warning Signals

Do low-battery devices exhibit higher anomaly rates?
This scatter plot tests the hypothesis that battery degradation
leads to sensor reliability issues.

In [ ]:
if health_df is not None:
    fig = px.scatter(
        health_pdf,
        x="avg_battery_level", y="anomaly_rate",
        color="risk_tier",
        color_discrete_map=tier_colors,
        size="health_score", size_max=12,
        hover_data=["device_id", "health_score"],
        opacity=0.7,
    )
    fig.update_layout(
        title="Battery Level vs Anomaly Rate (size = health score)",
        xaxis_title="Avg Battery Level (%)",
        yaxis_title="Anomaly Rate",
        template="plotly_white", height=450,
    )
    fig.show()

---
## Summary

This notebook demonstrated the complete pipeline lifecycle:

| Aspect | Key Insight |
|---|---|
| **Data Evolution** | Bronze captures everything; Silver cleans and enriches; Gold aggregates |
| **Anomaly Detection** | Multi-layered: threshold flags + z-scores + composite quality scoring |
| **Device Health** | Composite scoring identifies at-risk devices before they fail |
| **Delta Lake** | Full version history, time travel, and ACID guarantees at every layer |
| **Retention** | Bronze: 7d, Silver: 30d, Gold: indefinite — enforced by VACUUM |

In [ ]:
spark.stop()